<a href="https://colab.research.google.com/github/trainocate-japan/developing-agentic-ai-with-langchain/blob/main/chap02/hands-on/chap02_handson_2A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ハンズオン 2-A: Chat Completions API と Function Calling を動かす

**研修コース「LangChain による Agentic AI 開発実践」/ 第2章「LLM API の基礎」**

このハンズオンは、講師の解説を聞きながら**作成済みのセルを上から順に一緒に実行する**形式です。
受講者がコードを書く場面はありません (コードを書くのは演習 2-B です)。
まずは「動かして観察する」ことに集中してください。

## この Notebook で学ぶこと

**前半: Chat Completions API の基本 (1〜6)**

1. **最小の API 呼び出し** — `client.chat.completions.create()` でモデルに質問し、レスポンス構造を読む
2. **system プロンプト** — 1 行加えるだけでモデルの口調・役割が変わることを確認する
3. **ステートレス性** — API は会話を覚えていない、という本章最大の種明かしを実演する
4. **ミニチャットボット** — 履歴を `append` し続けると「覚えている」ように振る舞うことを確認する
5. **usage とコスト概算** — 消費トークン (`prompt_tokens` / `completion_tokens` / `total_tokens`) を読む
6. **パラメータ実験** — `temperature` で出力のばらつきを、`max_completion_tokens` で打ち切りを観察する

**後半: Function Calling 手動 1 周のデモ (7)**

7. **Function Calling デモ (`get_weather`)** — LLM に「行動」をさせる仕組みを、作成済みコードを実行して 4 ステップで体感する。`tool_calls` の受信 → `json.loads` → アプリ側で関数実行 → 結果を履歴に積んで最終応答、までを「動かして理解」する (受講者はコードを書きません。書くのは演習 2-B)

## 前提条件

- **Google アカウント**を持っていること
- このファイルを **Google Colab** で開いていること
- Colab の **[シークレット]** に `OPENAI_API_KEY` を登録済みであること (第1章の演習 1-1 で登録済みのはず。未登録でも、後述の「0-2. API キーのセットアップ」で登録できます)
- インターネット接続 (API を呼び出します)

## 所要時間

約 30 分 (前半の Chat Completions API 約 20 分 + 後半の Function Calling デモ 約 10 分。いずれも講師の解説を含む)

---
> **モデル名について**: 本教材ではモデル名を変数 `MODEL` に集約しています。教材中の例は `MODEL = "gpt-5.4"` ですが、**研修実施時には講師が指定する最新モデル名に差し替えてください**。1 箇所 (準備セル) を直すだけで全セルに反映されます。


## 0. セットアップ

### 0-1. 依存パッケージのインストール

OpenAI API を Python から呼び出すための `openai` パッケージをインストールします。
Colab には未インストール、または古いバージョンが入っていることがあるため、`-U` で最新へ更新します。

> 研修実施時は再現性のため、`openai==X.Y.Z` のようにバージョンをピン留めすることを推奨します
> (このセルを実行すると最新版が入るので、配布時点と挙動が変わる可能性があります)。


In [ ]:
# openai パッケージを最新版へインストール/更新
# 研修実施時は最新版にピン留め推奨 (例: !pip install -U "openai==X.Y.Z")
!pip install -U openai

### 0-2. API キーのセットアップ (Colab シークレット方式)

OpenAI API の呼び出しには **API キー**による認証が必要です。
API キーは「あなたのアカウントで課金してよい」という証明書のようなものなので、
**コードに直接書いてはいけません**。Colab では **[シークレット]** 機能で安全に管理します。

**操作手順** (未登録の場合):
1. 画面左のサイドバーにある **鍵アイコン 🔑 [シークレット]** をクリック
2. **[新しいシークレットを追加]** を押す
3. 名前に `OPENAI_API_KEY`、値にあなたの API キーを入力
4. このノートブックからのアクセスを **オン** にする

次のセルは、Colab のシークレットからキーを読み取り、環境変数 `OPENAI_API_KEY` に設定します。
`openai` パッケージはこの環境変数を自動的に読むため、以降のコードにキーは一切登場しません。
Colab 以外の環境 (ローカル等) では、あらかじめ環境変数 `OPENAI_API_KEY` を設定しておけば動きます。


In [ ]:
import os

# Colab のシークレットから API キーを読み込み、環境変数に設定する
# Colab 以外の環境では except 側に入り、既存の環境変数 OPENAI_API_KEY をそのまま使う
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab シークレットから OPENAI_API_KEY を読み込みました。")
except ImportError:
    # Colab 以外では環境変数 OPENAI_API_KEY が設定済みとみなす
    print("Colab 以外の環境です。環境変数 OPENAI_API_KEY を使用します。")

# キーが読めているか (存在のみ) を確認。キー本体は表示しない
print("APIキー設定済み:", bool(os.environ.get("OPENAI_API_KEY")))

### 0-3. クライアントとモデルの準備

`OpenAI` クラスをインスタンス化して `client` を作ります。
このとき環境変数 `OPENAI_API_KEY` が自動的に読み取られます。

モデル名は変数 `MODEL` に集約します。世代交代の速い領域なので、
「モデル名を 1 箇所で管理していつでも差し替えられるようにする」こと自体が実務の定石です。


In [ ]:
from openai import OpenAI

client = OpenAI()          # 環境変数 OPENAI_API_KEY からキーを自動取得
MODEL = "gpt-5.4"          # モデル名は変数に集約 (研修実施時に最新へ差し替え)

print("準備完了。使用モデル:", MODEL)

---

## 1. 最小の API 呼び出し

Chat Completions API の呼び出しは `client.chat.completions.create()` という 1 つのメソッドに集約されています。
最小の例を実行してみましょう。

**ここで注目すること**:
- 文字列を 1 つ渡すのではなく、`messages` という**辞書のリスト**を渡している
- 各辞書は `role` (誰の発言か) と `content` (発言の中身) を持つ
- チャット UI の入力欄に打ち込む文章は、API では `{"role": "user", "content": ...}` という 1 要素

**実行すると何が起きるか**: モデルが「LLM とは何か」を 1 文で答え、その回答テキストが表示されます。


In [ ]:
# 最小の API 呼び出し: user メッセージを 1 つだけ送る
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "LLM とは何ですか? 1 文で答えてください。"}
    ],
)

# 回答本体は choices[0].message.content に入っている
print(response.choices[0].message.content)

### レスポンス構造を読む

`response` オブジェクトの主要な構造は次のとおりです。

```text
response
├── choices[0]                  # 応答の候補 (通常は 1 件)
│   ├── message
│   │   ├── role: "assistant"   # モデルの発言であることを示す
│   │   ├── content: "..."      # 回答テキスト本体
│   │   └── tool_calls: None    # 後半の Function Calling で主役になるフィールド
│   └── finish_reason: "stop"   # 生成が終わった理由
└── usage                        # 消費トークン数 (このあと 5 で詳説)
    ├── prompt_tokens
    ├── completion_tokens
    └── total_tokens
```

- 回答本体は `response.choices[0].message.content`
- `choices` がリストなのは複数候補を要求できるため。通常は 1 件なので `[0]` 固定でよい
- `finish_reason` は「なぜ生成が終わったか」。正常終了なら `"stop"`

下のセルで、これらのフィールドを実際に取り出して確認します。
`finish_reason` がこのあと別の値 (`"length"` / `"tool_calls"`) になる場面が出てきます。


In [ ]:
# レスポンスの主要フィールドを取り出して確認する
choice = response.choices[0]

print("role          :", choice.message.role)          # => assistant
print("finish_reason :", choice.finish_reason)         # => stop (正常終了)
print("tool_calls    :", choice.message.tool_calls)    # => None (ツール未使用)
print("content       :", choice.message.content)
print("usage         :", response.usage)               # 消費トークン (5 で詳説)

---

## 2. system プロンプトで挙動を変える

`role` には `system` / `user` / `assistant` / `tool` の 4 種類があります。
そのうち **`system`** は、モデルの振る舞いや役割を指示するメッセージで、
ChatGPT の「カスタム指示」に相当します。

まずは分かりやすい例として、**関西弁で答えるアシスタント**にしてみます。
同じ質問でも、system メッセージ 1 行で出力の性格がガラリと変わることを観察してください。


In [ ]:
# system メッセージで口調を指定する (関西弁の例)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "あなたは関西弁で答えるアシスタントです。"},
        {"role": "user", "content": "LLM とは何ですか? 1 文で答えてください。"},
    ],
)
print(response.choices[0].message.content)  # 関西弁の回答が返る

### 実務での system プロンプト — ヘルプデスク担当の役割定義

実務では、口調だけでなく**アプリケーションの役割定義**を system に書きます。
本コースの演習ストーリーである「社内 IT ヘルプデスクエージェント」を先取りして、
**一次対応担当**としての役割を与えてみましょう (演習 2-B でこの延長線上のことをします)。

**実行すると何が起きるか**: モデルがヘルプデスク担当者の口調・スタンスで回答します。


In [ ]:
# 実務的な例: 社内 IT ヘルプデスクの一次対応担当という役割を与える
helpdesk_system = (
    "あなたは社内 IT ヘルプデスクの一次対応担当です。"
    "社内システムに関する質問に、丁寧かつ簡潔に答えてください。"
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": helpdesk_system},
        {"role": "user", "content": "VPN に接続できません。どうすればよいですか?"},
    ],
)
print(response.choices[0].message.content)

---

## 3. ステートレス性の確認 — 本章最大の種明かし

ここが本章の核心です。**API はステートレス**、つまり呼び出しごとの状態を一切保持しません。
各リクエストは独立しており、モデルは渡された `messages` 配列の中身だけを見て応答します。

次の 2 つの呼び出しを順に行うと何が起きるか、**予想しながら**実行してください。

1. 1 回目: 名前を伝える
2. 2 回目: **新規の呼び出し**で名前を聞く

**期待される結果**: 2 回目でモデルは名前を答えられません。1 回目の会話を完全に忘れているからです。


In [ ]:
# 1 回目: 名前を伝える
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "私の名前はカレンです。覚えてくださいね。"}],
)
print("【1回目】", r1.choices[0].message.content)  # 「わかりました、カレンさん」等

# 2 回目: 新規の呼び出しで名前を聞く (1回目とは別の messages を渡している点に注目)
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "私の名前は何でしたか?"}],
)
print("【2回目】", r2.choices[0].message.content)  # 「お名前を伺っていません」等

2 回目で名前を答えられなかったはずです。これが「ステートレス」です。

では、ChatGPT はなぜ会話を覚えているように見えるのでしょうか。
答えは単純で、**アプリケーション側が会話履歴を保存しておき、毎回のリクエストで過去の全メッセージを
`messages` 配列に積んで送信している**からです。
「会話」とはサーバー側のセッションではなく、クライアントが毎回全文を再送する申告の積み重ねです。

次の 4 で、その「全履歴を積む」仕組みを実際に作ります。


---

## 4. ミニチャットボット — 履歴を積めば「覚える」

`messages` リストに user と assistant の発言を交互に `append` しながら会話を続ければ、
モデルは「覚えている」ように振る舞います。十数行ですが、これが「会話できる AI アプリ」の最小の本体です。

### 4-(a) 対話版

下のセルは `input()` でユーザー入力を受け取る**対話版**です。Colab 上で実行するとセル下部に入力欄が出ます。
`quit` と入力すると終了します。

> **注意**: このセルは手入力が必要なため、`input()` の応答待ちでセルが止まります。
> 講師のデモで一度動かしたら停止 (■ ボタン) し、次の 4-(b) スクリプト版で「覚える」様子を自動再現します。
> 自分のペースで進める場合、このセルは飛ばして 4-(b) に進んでも構いません。


In [ ]:
# 対話版ミニチャットボット。input() で手入力するため自動実行はできない。
# 試すときは「私の名前はカレンです」→「私の名前は?」と入力すると、覚えていることを確認できる。
messages = [
    {"role": "system", "content": "あなたは親切なアシスタントです。"}
]

while True:
    user_input = input("あなた> ")
    if user_input == "quit":
        break

    # ① ユーザー発言を履歴に積む
    messages.append({"role": "user", "content": user_input})

    # ② 「全履歴」を送信する (ここがポイント)
    response = client.chat.completions.create(model=MODEL, messages=messages)
    assistant_message = response.choices[0].message.content
    print(f"AI> {assistant_message}")

    # ③ モデルの応答も履歴に積む (積み忘れると次のターンで文脈が壊れる)
    messages.append({"role": "assistant", "content": assistant_message})

### 4-(b) スクリプト版 — 手入力なしで「覚える」を再現

対話版は手入力が必要で自動実行できません。そこで、あらかじめ用意した発言リスト `turns` を
順に処理する**スクリプト版**を用意しました。中身のロジックは対話版とまったく同じ
(履歴に積む → 全履歴を送信 → 応答も積む) で、`input()` を `turns` のループに置き換えただけです。

`turns` の 1 つ目で名前を伝え、2 つ目で名前を尋ねます。
**履歴を積んでいるので、2 ターン目でモデルは名前を覚えている**はずです (3 のステートレス実験との対比)。

**実行すると何が起きるか**: 各ターンの「あなた」「AI」のやり取りが順に表示され、
最後のターンで AI が「カレン」という名前を答えます。


In [ ]:
# スクリプト版ミニチャットボット: input() の代わりに turns を順に処理する。
# ロジックは対話版と同一。履歴 messages を積み続けることで「覚える」様子を手入力なしで再現する。
turns = ["私の名前はカレンです", "私の名前は?"]

messages = [
    {"role": "system", "content": "あなたは親切なアシスタントです。"}
]

for user_input in turns:
    print(f"あなた> {user_input}")

    # ① ユーザー発言を履歴に積む
    messages.append({"role": "user", "content": user_input})

    # ② 「全履歴」を送信する
    response = client.chat.completions.create(model=MODEL, messages=messages)
    assistant_message = response.choices[0].message.content
    print(f"AI> {assistant_message}\n")

    # ③ モデルの応答も履歴に積む
    messages.append({"role": "assistant", "content": assistant_message})

# 最終的に messages に何件積まれたか (system + user/assistant が交互に積まれている)
print(f"--- 会話終了: messages には {len(messages)} 件のメッセージが積まれています ---")

---

## 5. usage とコスト概算

レスポンスに含まれる `usage` には、消費したトークン数の 3 つの数字が入っています。

- `prompt_tokens` — 入力 (送信した `messages` 全体) のトークン数
- `completion_tokens` — 出力 (モデルが生成した応答) のトークン数
- `total_tokens` — 両者の合計

API の課金はこのトークン数に単価を掛けて決まります。重要なのは
**入力と出力で単価が異なる** (一般に出力のほうが高い) ことです。

まずは単発の呼び出しで `usage` を表示してみましょう。


In [ ]:
# 単発呼び出しで usage を表示する
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "LLM の歴史を簡単に説明してください。"}],
)

usage = response.usage
print("usage 全体     :", usage)
print("prompt_tokens  :", usage.prompt_tokens)       # 入力トークン
print("completion_tokens:", usage.completion_tokens) # 出力トークン
print("total_tokens   :", usage.total_tokens)        # 合計

### ターンが進むと prompt_tokens はどうなるか

3 で学んだとおり、履歴は毎ターン**全送信**されます。
ということは、チャットボットでターンが進むほど、送信する `messages` が膨らみ、
`prompt_tokens` は**単調に増加**していくはずです。

下のセルでは、ミニチャットボットを複数ターン回しながら、各ターンの `prompt_tokens` を記録します。
**実行すると何が起きるか**: ターンが進むごとに `prompt_tokens` が増えていく様子が表で確認できます。
これが「履歴を全部送る素朴な戦略はいずれコンテキスト上限に達する」という問題の正体です
(この問題への答え——履歴の trim や要約——は第4章で学びます)。


In [ ]:
# ターンを重ねて prompt_tokens の増加を観察する
turns = [
    "こんにちは。私は経理部のカレンです。",
    "経費精算システムの使い方を教えてください。",
    "ありがとう。ついでに勤怠システムの場所も教えて。",
]

messages = [{"role": "system", "content": "あなたは社内 IT ヘルプデスクの一次対応担当です。"}]

print(f"{'ターン':<6}{'prompt_tokens':<16}{'completion_tokens':<18}{'total_tokens'}")
print("-" * 56)

for i, user_input in enumerate(turns, start=1):
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    u = response.usage
    print(f"{i:<6}{u.prompt_tokens:<16}{u.completion_tokens:<18}{u.total_tokens}")
    # 応答も履歴に積む (次ターンの入力に含まれるため prompt_tokens が増える要因になる)
    messages.append({"role": "assistant", "content": response.choices[0].message.content})

print("\n観察: ターンが進むほど prompt_tokens が増えていれば、履歴の全送信が効いている証拠です。")

### コスト概算のしかた (フェルミ推定)

`usage` の 3 つの数字と料金表 (「100 万トークンあたり何ドル」で公表) があれば、
1 リクエストのコストが概算できます。フェルミ推定の例:

- 想定: 社内チャットボットが 1 日 1,000 会話、平均 10 ターン
- 1 会話あたり 入力 20,000 トークン・出力 3,000 トークン程度とすると…
- 1 日: 入力 2,000 万・出力 300 万トークン
- 1 か月 (20 営業日): 入力 4 億・出力 6,000 万トークン
- あとは使用モデルの単価を掛ければ月額が出る

下のセルは、**単価を仮に置いて**月額を概算する電卓です (単価はモデルにより桁が変わるため、
ここでは仮の値です。実際の単価は使用モデルの料金ページで確認してください)。


In [ ]:
# コスト概算の電卓 (単価は仮の値。実際は使用モデルの料金表で置き換える)
# 料金は「100 万トークンあたり米ドル」で公表されるのが一般的
price_input_per_1m = 1.00    # 入力 100 万トークンあたり (仮)
price_output_per_1m = 3.00   # 出力 100 万トークンあたり (仮。一般に入力より高い)

# フェルミ推定の想定値 (上の Markdown の試算より)
monthly_input_tokens = 400_000_000    # 月 4 億トークン (入力)
monthly_output_tokens = 60_000_000    # 月 6,000 万トークン (出力)

cost_input = monthly_input_tokens / 1_000_000 * price_input_per_1m
cost_output = monthly_output_tokens / 1_000_000 * price_output_per_1m

print(f"入力コスト概算: ${cost_input:,.2f}")
print(f"出力コスト概算: ${cost_output:,.2f}")
print(f"月額概算 (合計): ${cost_input + cost_output:,.2f}")
print("\n※ 単価は仮の値です。大事なのは『usage の 3 つの数字だけでこの計算ができる』ことです。")

---

## 6. パラメータ実験

`create()` には `model` と `messages` 以外にも、生成を制御するパラメータを渡せます。
代表的な 2 つを実験します。

### 6-1. temperature — 出力の多様性

`temperature` は出力の多様性 (ランダム性) を制御します。
- `0` にするとほぼ決定的な (毎回同じに近い) 出力
- 高くするほど多様で意外性のある出力

`0` と `1.2` で、それぞれ 3 回ずつ俳句を詠ませて比較します。

**期待される結果**: `temperature=0` では 3 回ともほぼ同じ句が、`1.2` では毎回違う句が返ります。


In [ ]:
# temperature を 0 と 1.2 で各 3 回実行し、出力のばらつきを比較する
for temp in [0, 1.2]:
    print(f"--- temperature={temp} ---")
    for _ in range(3):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "AI をテーマに俳句を 1 句詠んでください。"}],
            temperature=temp,   # 0: ほぼ決定的 / 高め: 多様
        )
        print(r.choices[0].message.content)
    print()

事実の照会や分類のように**再現性が欲しいタスクでは低く**、
ブレインストーミングのような**発想系では高く**、が使い分けの目安です。

> **補足**: reasoning 系モデルでは `temperature` などが指定不可の場合があります。
> モデルを切り替えたら、使えるパラメータの互換性を確認する習慣をつけてください。
> もしこのセルでパラメータ関連のエラーが出たら、それは「このモデルは temperature 非対応」のサインです。

### 6-2. max_completion_tokens — 出力の打ち切りと finish_reason

`max_completion_tokens` は出力トークン数の上限です。
これを極端に小さくすると、モデルは文章の途中で打ち切られます。
このとき `finish_reason` が `"stop"` ではなく **`"length"`** になります。

**期待される結果**: 回答テキストが途中で切れ、`finish_reason` が `"length"` と表示されます。


In [ ]:
# max_completion_tokens を極端に小さくして、打ち切り (finish_reason="length") を観察する
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "LLM の歴史を詳しく説明してください。"}],
    max_completion_tokens=20,   # 極端に小さくして打ち切りを観察
)
print("content       :", r.choices[0].message.content)   # 途中で切れた文章
print("finish_reason :", r.choices[0].finish_reason)     # => "length" (正常終了なら "stop")

「応答が途中で切れている」という不具合報告を受けたら、**まず `finish_reason` を確認する**——
これが API 開発のデバッグの第一歩です。
`"stop"` なら正常終了、`"length"` なら上限による打ち切りです。

このあと後半のセクション 7 では、`finish_reason` がもう 1 つの値 **`"tool_calls"`** になる場面に出会います。
この小さなフィールドが、実はエージェント開発の分岐点になります。


---

## 7. Function Calling 手動 1 周のデモ (get_weather)

ここからが後半です。**作成済みのコードを実行して、LLM に「行動」をさせる仕組み (Function Calling) の 1 周を体感**します。
受講者はコードを書きません (書く側は次の演習 2-B です)。「動かして理解する」ことに集中してください。

### なぜ Function Calling が必要か

LLM 単体には、明確な限界が 3 つあります。

- **リアルタイムの情報を知らない** (学習データの時点で知識が止まっている)
- **厳密な計算が苦手** (もっともらしい数字を「生成」してしまう)
- **外部システムを操作できない** (テキストを出すことしかできない)

「今日の東京の天気は?」と聞かれても、モデル自身には知りようがありません。
この限界を破り、第1章で学んだ ReAct ループの「**行動**」を実現する仕組みが **Function Calling** です。
アプリ側が「こういう関数を用意してあります」とモデルに伝え、モデルが必要に応じてその関数を「呼び出す」——その現代的な標準実装です。

### 最重要ポイント: LLM は関数を実行しない

先に、このセクション最大のポイントを宣言します。

> **LLM は関数を実行しません。実行するのは常にアプリ側 (皆さんのコード) です。**

「Function Calling」「LLM がツールを呼び出す」という言い回しは比喩です。モデルがするのは
「この関数をこの引数で実行したい」という**意図を JSON で宣言する**ことだけ。
実際に関数を実行するのは、紛れもなく皆さんの Python コードです。
実行するかどうか・どう実行するかの主導権が完全にアプリ側にあるからこそ、
危険な操作の前に人間の承認を挟む (Human-in-the-Loop、第6章) ことが可能になります。

### 仕組みの全体像 — 4 ステップ

Function Calling の 1 周は、**アプリ ⇔ OpenAI API ⇔ ローカル関数 (`get_weather`)** の 3 者の間を情報が流れる 4 ステップで構成されます。
このあと、この 4 ステップを 1 つずつコードで実行していきます。

```text
[ステップ①] アプリ  --- messages + tools (関数の名前・説明・引数スキーマ) --->  API
            (「東京の天気は?」を tools 付きで送信)

[ステップ②] API  --- finish_reason="tool_calls" / tool_calls=[get_weather + 引数(JSON文字列)] --->  アプリ
            (モデルは答える代わりに「get_weather を location="東京" で呼びたい」と宣言)

[ステップ③] アプリ  --- get_weather(location="東京") --->  ローカル関数
            ローカル関数  --- "晴れ、気温 24 度" --->  アプリ
            (json.loads で引数をパースし、アプリ側が関数を実行)

[ステップ④] アプリ  --- messages + assistant(tool_calls入り) + tool(実行結果) --->  API
            API  --- 「東京は晴れで 24 度です」 --->  アプリ
            (実行結果を role:"tool" で履歴に積んで再送信 → 最終応答)
```

要点を 2 つ、先に押さえておきます。

- **`arguments` は dict ではなく JSON 文字列**です (ステップ②で確認します)。
- **履歴は「宣言 → 結果」のペアで積む** (ステップ④で、assistant メッセージ → tool メッセージの順に積みます)。


### ステップ①: ツールを定義してリクエストする

まず、モデルに教える関数を用意します。題材は天気を返す関数です
(本物の気象 API は使わず、固定値を返す**ダミー実装**にします。仕組みの学習にはこれで十分です)。

次に、この関数の存在をモデルに伝えるための**ツール定義**を書きます。
ツール定義は「関数の取扱説明書」を JSON で書いたもので、`parameters` の部分は
**JSON Schema** (JSON データの構造を記述する標準形式) で引数の仕様を書きます。

ここで特に注目してほしいのが **`description`** です。モデルは皆さんの Python コードの中身を見られません。
**ツールを使うか・どのツールを使うかを判断する材料は、この description (と関数名・引数スキーマ) だけ**です。
だから丁寧に書きます (第3章では `@tool` 関数の docstring がこの description に変換されることを学びます)。

下のセルを実行して、`get_weather` 関数とツール定義を準備します。


In [ ]:
import json


# モデルに「呼び出させる」関数。本物の気象 API は使わず、固定値を返すダミー実装
def get_weather(location: str) -> str:
    """指定された場所の天気を返す (ダミー実装)"""
    return f"{location}の天気: 晴れ、気温 24 度"


# この関数の存在をモデルに伝えるツール定義 (関数の取扱説明書を JSON で書く)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            # description はモデルがツールを選ぶ唯一の判断材料。丁寧に書く
            "description": "指定された場所の現在の天気を取得する",
            "parameters": {                  # 引数の仕様を JSON Schema で記述
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "天気を知りたい場所。例: 東京",
                    }
                },
                "required": ["location"],    # 必須の引数
            },
        },
    }
]

# 関数単体の動作確認
print(get_weather("東京"))
print("ツール定義:", tools[0]["function"]["name"])

### ステップ②: tool_calls を受け取って観察する

定義ができたので、`tools` を添えて「**東京の天気は?**」と送ります。
モデルが「この質問にはツールが必要だ」と判断すると、レスポンスの様子がこれまでと一変します。

下のセルで、次の 3 点を観察してください。

- `finish_reason` が `"stop"` ではなく **`"tool_calls"`** になる
- `message.content` の代わりに `message.tool_calls` に**呼び出し宣言**が入る
- `tool_calls[0]` の中身: `.function.name` (呼びたい関数名) / `.function.arguments` / `.id`

特に大事なのが **`.function.arguments`** です。**これは dict ではなく JSON「文字列」**です。
見た目は dict そっくりですが、`tool_call.function.arguments["location"]` と書くと
「文字列に文字列インデックスは使えない」という TypeError になります。
ステップ③で必ず `json.loads()` でパースします。


In [ ]:
# 「東京の天気は?」を tools 付きで送信する
messages = [{"role": "user", "content": "東京の天気は?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,          # ツール定義を添える
)

# finish_reason が "stop" ではなく "tool_calls" になる
print("finish_reason :", response.choices[0].finish_reason)   # => "tool_calls"

# 呼び出し宣言を取り出す。tool_calls はリストなので [0] で最初の宣言を取る
tool_call = response.choices[0].message.tool_calls[0]

print("id            :", tool_call.id)                  # 例: "call_abc123" (④で必要になる ID)
print("function.name :", tool_call.function.name)       # => "get_weather"
print("arguments     :", tool_call.function.arguments)  # => '{"location": "東京"}' ← JSON「文字列」
print("arguments の型 :", type(tool_call.function.arguments))  # => <class 'str'> (dict ではない!)

### ステップ③: アプリ側で関数を実行する

宣言を読み取ったら、実行するのは**アプリ側の仕事**です。
`arguments` を `json.loads` で dict に変換し、`**args` で関数に渡します。

下の 2 行こそが、このセクション冒頭で宣言した「**LLM は関数を実行しない**」の動かぬ証拠です。
実行しているのは紛れもなく皆さんの Python コードです。
ここで結果を検査することも、危険なら実行を拒否することも、人間の承認を待つことも、すべてアプリ側の自由です。


In [ ]:
# arguments (JSON 文字列) を dict にパースしてから、アプリ側で関数を実行する
args = json.loads(tool_call.function.arguments)   # JSON 文字列 → dict
print("パース後の引数 (dict):", args)               # => {"location": "東京"}

# ↓ この行が「LLM は実行しない」の証拠。実行しているのはアプリ (あなたのコード)
result = get_weather(**args)
print("関数の実行結果       :", result)             # => "東京の天気: 晴れ、気温 24 度"

### ステップ④: 結果を返して最終応答を得る

最後に、実行結果をモデルに伝えて最終応答をもらいます。
履歴 `messages` には **2 つ**のメッセージを**正しい順序**で積む必要があります。

1. **(a) 先に** `tool_calls` 入りの assistant メッセージを積む
   - これは `response.choices[0].message` (= モデルが返したメッセージ) **そのもの**です
   - **積み忘れに注意!** これを飛ばすと API がエラーを返します
2. **(b) 次に** 実行結果を `role:"tool"` で積む
   - `tool_call_id` で「どの呼び出し宣言への答えか」を対応付けます (= ②で受け取った `tool_call.id`)

> **なぜ順序が大事か**: `role:"tool"` のメッセージは「直前に `tool_calls` を持つメッセージへの応答」でなければならない、
> というルールがあります。(a) を忘れたり (b) を先に積むと
> `BadRequestError: messages with role 'tool' must be a response to a preceding message with 'tool_calls'`
> になります。**履歴は「宣言 → 結果」のペアで積む**、と覚えてください。

積み終えたら全履歴を再送信します。今度はモデルが**実行結果を踏まえた最終応答**を返します。


In [ ]:
# (a) tool_calls 入りの assistant メッセージを先に積む (積み忘れ注意!)
messages.append(response.choices[0].message)

# (b) 実行結果を role:"tool" で積む。tool_call_id でどの呼び出しへの答えかを示す
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,   # ②で受け取った id をそのまま使う
    "content": result,
})

# 全履歴を再送信 → モデルが結果を踏まえた最終応答を生成
final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
print("最終応答:", final.choices[0].message.content)
# => 例: "東京は晴れで、気温は 24 度です。"

### ツールが不要なときは tool_calls が返らない

ここまでで Function Calling の 1 周が完成しました。最後に、**モデルがツールの要否を自分で判断している**ことを確認します。

`tools` を添えても、ツールが必要ない質問ではモデルは `tool_calls` を返さず、普通に `content` で答えます。
「**こんにちは**」を送って確かめましょう。

**期待される結果**: `finish_reason` は `"stop"` (= `"tool_calls"` ではない)、`tool_calls` は `None`、
`content` に挨拶の返事が入ります。`tools` を渡すと毎回ツールが呼ばれるわけではない——
モデルが「これはツール不要」と判断できている証拠です。


In [ ]:
# ツール不要の質問では tool_calls が返らないことを確認する
greeting = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "こんにちは"}],
    tools=tools,          # tools を添えても、不要ならモデルは使わない
)

print("finish_reason :", greeting.choices[0].finish_reason)        # => "stop" ("tool_calls" ではない)
print("tool_calls    :", greeting.choices[0].message.tool_calls)   # => None
print("content       :", greeting.choices[0].message.content)      # => 挨拶の返事

### このデモのまとめ → 演習 2-B へ

この `get_weather` デモで、Function Calling の 4 ステップを動かして確認しました。

| ステップ | 何をしたか | キーポイント |
|---|---|---|
| ① ツール定義 + リクエスト | `tools` (JSON Schema 込み) を添えて「東京の天気は?」を送信 | `description` がツール選択の唯一の判断材料 |
| ② tool_calls 受信 | `finish_reason="tool_calls"`、`tool_calls[0]` を観察 | `arguments` は dict ではなく **JSON 文字列** |
| ③ アプリ側で実行 | `json.loads` → `get_weather(**args)` | **関数を実行するのはアプリ (LLM ではない)** |
| ④ 結果を返す | assistant (tool_calls 入り) → tool (`tool_call_id` 一致) を積み再送信 | 履歴は「宣言 → 結果」のペアで積む |

そして「こんにちは」では `tool_calls` が返らない (モデルがツール不要と判断する) ことも確認しました。

**この `get_weather` デモで見た 4 ステップを、次の演習 2-B では自分の手で helpdesk の `get_system_status` に実装します。**
動かして理解した仕組みを、今度は `# TODO` を埋めながら自分で組み立ててみましょう。


---

## まとめ — ハンズオン 2-A で確認したこと

| # | 確認したこと | キーポイント |
|---|---|---|
| 1 | 最小の API 呼び出し | `messages` は辞書のリスト。回答は `choices[0].message.content` |
| 2 | system プロンプト | 1 行で口調・役割が変わる。実務ではアプリの役割定義を書く |
| 3 | ステートレス性 | API は会話を覚えない。各リクエストは独立 |
| 4 | ミニチャットボット | 履歴を `append` し全送信すれば「覚える」ように振る舞う |
| 5 | usage とコスト | 3 つの数字でコスト概算。ターンが進むと `prompt_tokens` 増加 |
| 6 | パラメータ | `temperature` で多様性、`max_completion_tokens` で打ち切り (`finish_reason="length"`) |
| 7 | **Function Calling デモ** | `tool_calls` 受信 → `json.loads` → アプリ側で実行 → tool ロールで結果返却の 4 ステップ。**関数を実行するのはアプリ** |

### 次は演習 2-B へ

この Notebook と**同じ環境**で、演習 2-B に進みます。
セクション 7 の `get_weather` デモで動かして理解した Function Calling の 4 ステップを、
今度は社内 IT ヘルプデスクの稼働状況に答える `get_system_status` ツールで、
**自分の手で 1 周**させます (`# TODO` を埋める形式)。
デモで観察した `finish_reason="tool_calls"` や `tool_call_id` を、今度は自分のコードで扱います。お楽しみに。
